# YapIndex Retrieval Evaluation

This notebook compares retrieval **before** and **after** reranking.

## Before you run

1. Open this project in a **WSL VS Code window**.
2. Select the kernel **YapIndex WSL GPU**.
3. Run the cells from top to bottom.

The first run downloads the reranker model and may take several minutes. Set `LIMIT = 5` in the setup cell for a quick test. Set it to `None` to evaluate all 70 questions.

## Step 2: Configure the GPU run

If the dependency check passes, continue to the next code cell. The reranker uses CUDA; FAISS performs the initial search.

## Step 1: Confirm dependencies

The **YapIndex WSL GPU** kernel already contains the dependencies. Run the next cell once to check them. Do not install packages from the notebook.

In [ ]:
import sys

print(f"Python: {sys.executable}")

import torch
import faiss

print("Dependencies: OK")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select the 'YapIndex WSL GPU' kernel."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"FAISS GPUs: {faiss.get_num_gpus()}")

In [ ]:
import json
import sys
from pathlib import Path

import torch

# Find the project root when this notebook is opened from the notebooks folder.
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "utils").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import utils.faiss_store
importlib.reload(utils.faiss_store)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select the 'YapIndex WSL GPU' kernel."
    )

from utils.faiss_store import load_vectorstore
from utils.reranker import RERANKER_MODEL, load_reranker, rerank_results

RETRIEVAL_K = 20
LIMIT = None  # Change to None to evaluate all questions.
QUESTIONS_PATH = PROJECT_ROOT / "evals" / "meridian_rag_eval.jsonl"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Project: {PROJECT_ROOT}")
print(f"Questions: {LIMIT if LIMIT is not None else 'all'}")

## Step 3: Load questions, FAISS, and reranker

In [18]:
with QUESTIONS_PATH.open(encoding="utf-8") as file:
    questions = [json.loads(line) for line in file if line.strip()]

if LIMIT is not None:
    questions = questions[:LIMIT]

print(f"Loaded {len(questions)} question(s).")
print("Loading FAISS index...")
vectorstore = load_vectorstore(str(PROJECT_ROOT / "faiss_index"), use_gpu=True)
print(f"FAISS index: {type(vectorstore.index).__name__}")

print("Loading reranker on CUDA...")
reranker = load_reranker(model_name=RERANKER_MODEL, device="cuda")
print(f"Reranker: {RERANKER_MODEL}")
print(f"Device: {reranker.device}")

Loaded 70 question(s).
Loading FAISS index...


FAISS index: GpuIndexFlat
Loading reranker on CUDA...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1244.65it/s]


Reranker: BAAI/bge-reranker-v2-m3
Device: cuda:0


## Step 4: Define scoring helpers

In [19]:
def normalize_source(source: str) -> str:
    return source.replace("\\", "/").lstrip("./")


def get_sources(documents) -> list[str]:
    return [
        normalize_source(document.metadata.get("source", "Unknown"))
        for document in documents
    ]


def score_results(documents, question: dict) -> dict[str, int]:
    sources = get_sources(documents)
    gold_sources = {
        normalize_source(source) for source in question["gold_sources"]
    }
    distractor_sources = {
        normalize_source(source)
        for source in question.get("distractor_sources", [])
    }

    return {
        "hit_at_3": int(bool(gold_sources & set(sources[:3]))),
        "hit_at_5": int(bool(gold_sources & set(sources[:5]))),
        "hit_at_10": int(bool(gold_sources & set(sources[:10]))),
        "distractor_at_5": int(
            bool(distractor_sources & set(sources[:5]))
        ),
    }


def add_scores(totals: dict[str, int], scores: dict[str, int]) -> None:
    for key, value in scores.items():
        totals[key] += value


def empty_totals() -> dict[str, int]:
    return {
        "hit_at_3": 0,
        "hit_at_5": 0,
        "hit_at_10": 0,
        "distractor_at_5": 0,
    }

## Step 5: Run evaluation

In [20]:
before_totals = empty_totals()
after_totals = empty_totals()

for index, question in enumerate(questions, start=1):
    print(f"Evaluating {index}/{len(questions)}...", end="\r", flush=True)

    before_results = vectorstore.similarity_search(
        question["question"],
        k=RETRIEVAL_K,
    )
    after_results = rerank_results(
        reranker,
        question["question"],
        before_results,
    )

    add_scores(before_totals, score_results(before_results, question))
    add_scores(after_totals, score_results(after_results, question))

print()

Evaluating 70/70...


## Step 6: Compare results

In [22]:
def print_metrics(label: str, totals: dict[str, int], count: int) -> None:
    print(f"\n{label}")
    print("-" * len(label))
    print(f"Recall@3: {totals['hit_at_3'] / count:.2%}")
    print(f"Recall@5: {totals['hit_at_5'] / count:.2%}")
    print(f"Recall@10: {totals['hit_at_10'] / count:.2%}")
    print(f"Distractor rate@5: {totals['distractor_at_5'] / count:.2%}")


print(f"Evaluated {len(questions)} question(s).")
print_metrics("Before reranking", before_totals, len(questions))
print_metrics("After reranking", after_totals, len(questions))

Evaluated 70 question(s).

Before reranking
----------------
Recall@3: 58.57%
Recall@5: 67.14%
Recall@10: 74.29%
Distractor rate@5: 17.14%

After reranking
---------------
Recall@3: 65.71%
Recall@5: 77.14%
Recall@10: 80.00%
Distractor rate@5: 17.14%
